# SNNAI v6.0 Large-Scale SNN LM + Transformer Comparison

This notebook trains a large SNNAI character-level language model on a real corpus and compares it against a small Transformer baseline.

- v5.6: large-scale architecture
- v5.7: large-corpus training
- v5.8: Transformer comparison
- v5.9: quantization / pruning
- v6.0: Transformer-level challenge evaluation

In [ ]:
# Install dependencies
# CUDA 12.6 build keeps P100 (sm_60) and T4 (sm_75) compatibility
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu126
!pip install -q snntorch numpy requests

In [ ]:
# Clone SNNAI repository
!rm -rf snnai
!git clone --depth 1 --branch v6.2.1 https://github.com/sabunosuke1008-create/snnai.git
import sys
sys.path.insert(0, 'snnai')

In [ ]:
# Download tiny Shakespeare corpus
import requests, os, time, json
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
corpus = requests.get(url).text
print('Corpus length:', len(corpus))

In [ ]:
# Imports
import torch
import torch.nn as nn
from snnai.modules.language import CharTokenizer, SpikeEncoder
from snnai.modules.language.large_scale_lm import LargeScaleSNNLM, count_parameters
from snnai.benchmarks.transformer_comparison import TransformerBaseline, compare_models
from snnai.benchmarks.optimization import quantize_weights, prune_weights
from snnai.benchmarks.energy_quantification import quantize_energy

In [ ]:
# Optionally download a larger corpus (WikiText-2) for better generalization
import os
if not os.path.exists('/kaggle/working/wikitext-2-raw-v1/test.txt'):
    !wget -q https://s3.amazonaws.com/research.metamind.io/wikitext/wikitext-2-raw-v1.zip
    !unzip -q wikitext-2-raw-v1.zip -d /kaggle/working/
    print('Downloaded WikiText-2')
else:
    print('WikiText-2 already present')

# Combine Tiny Shakespeare and WikiText-2 train+valid+test for a larger corpus
wt_files = [
    '/kaggle/working/wikitext-2-raw-v1/wiki.train.raw',
    '/kaggle/working/wikitext-2-raw-v1/wiki.valid.raw',
    '/kaggle/working/wikitext-2-raw-v1/wiki.test.raw',
]
wt_text = ''.join(open(f).read() for f in wt_files if os.path.exists(f))
if wt_text:
    corpus = corpus + '\n' + wt_text
    print('Combined corpus length:', len(corpus))
else:
    print('Using Tiny Shakespeare only, length:', len(corpus))

# Tokenizer and dataset
vocab = sorted(set(corpus))
tokenizer = CharTokenizer(''.join(vocab))
data = torch.tensor([tokenizer.encode(corpus)], dtype=torch.long)
print('Vocab size:', tokenizer.vocab_size)
print('Data shape:', data.shape)


In [ ]:
# Training helpers
def make_batch(data, batch_size, seq_len, device):
    starts = torch.randint(0, data.size(1) - seq_len - 1, (batch_size,))
    inputs = torch.stack([data[0, s:s+seq_len] for s in starts])
    targets = torch.stack([data[0, s+1:s+seq_len+1] for s in starts])
    return inputs.to(device), targets.to(device)

def perplexity(logits, targets):
    loss = nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
    return torch.exp(loss).item()

## Train large SNNAI model

In [ ]:
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    if cap[0] < 7:
        device = torch.device('cpu')
        print(f'GPU compute capability {cap} is below PyTorch minimum (7.0); using CPU')
    else:
        device = torch.device('cuda')
        print('Using device:', device)
else:
    device = torch.device('cpu')
    print('Using device:', device)

# Adjust dimensions for Kaggle T4; shrink on CPU fallback to fit timeout
if device.type == 'cpu':
    embed_dim = 64
    hidden_dim = 256
    num_layers = 2
    seq_len = 64
    batch_size = 16
    time_steps = 10
    epochs = 5
    print('Using small config for CPU fallback to avoid timeout')
else:
    # Smaller model to avoid memorizing the small corpus
    embed_dim = 128
    hidden_dim = 512
    num_layers = 3
    seq_len = 128
    batch_size = 32
    time_steps = 20
    epochs = 20
lr = 5e-4
label_smoothing = 0.1

snn_model = LargeScaleSNNLM(vocab_size=tokenizer.vocab_size, embed_dim=embed_dim,
                            hidden_dim=hidden_dim, num_layers=num_layers,
                            dropout=0.2,
                            output_mode='mem_last').to(device)
encoder = SpikeEncoder(vocab_size=tokenizer.vocab_size, time_steps=time_steps)
optimizer = torch.optim.AdamW(snn_model.parameters(), lr=lr, weight_decay=0.01)
print('SNN params:', count_parameters(snn_model))


In [ ]:
# Use the improved trainer with validation split, LR schedule, and gradient clipping
from snnai.benchmarks.large_corpus_trainer import LargeCorpusTrainer, WarmupCosineSchedule
from snnai.benchmarks.diagnostic import SNNDiagnostic

snn_model = LargeScaleSNNLM(vocab_size=tokenizer.vocab_size, embed_dim=embed_dim,
                            hidden_dim=hidden_dim, num_layers=num_layers,
                            output_mode='mem_mean', dropout=0.2).to(device)
optimizer = torch.optim.AdamW(snn_model.parameters(), lr=lr, weight_decay=0.01)
total_steps = epochs * max(1, len(data) // (batch_size * seq_len))
scheduler = WarmupCosineSchedule(optimizer, warmup_steps=max(1, total_steps // 10),
                                 total_steps=max(1, total_steps), base_lr=lr, min_lr=1e-5)
trainer = LargeCorpusTrainer(snn_model, optimizer, tokenizer, device=device,
                             val_ratio=0.05, max_grad_norm=1.0, scheduler=scheduler,
                             split_mode='temporal', label_smoothing=label_smoothing)

# Attach diagnostic hooks for the first few steps to inspect gradients and spikes
diag = SNNDiagnostic(snn_model)
history = trainer.train(data, epochs=epochs, seq_len=seq_len, batch_size=batch_size,
                        time_steps=time_steps, save_path='snnai_v6_large_lm.pt')
diag.to_json('snn_diagnostics.json')
snn_history = [{'epoch': i, 'loss': history['train_loss'][i], 'ppl': history['train_ppl'][i]}
               for i in range(len(history['train_loss']))]


## Train Transformer baseline

In [ ]:
# Transformer baseline is now trained inside fair_compare() together with the SNN.
# This ensures identical data splits, epochs, learning rate schedule, and gradient clipping.
from snnai.benchmarks.transformer_comparison import TransformerBaseline, fair_compare

transformer_model = TransformerBaseline(vocab_size=tokenizer.vocab_size, d_model=256,
                                        nhead=4, num_layers=4, dim_feedforward=1024).to(device)


## Compare latency, parameters, and energy

In [ ]:
# Fair comparison: train both models on the same data and report full metrics
comparison_results = fair_compare(data, tokenizer, snn_model, transformer_model,
                                  epochs=epochs, seq_len=seq_len, batch_size=batch_size,
                                  time_steps=time_steps, device=device, lr=lr,
                                  save_dir='/kaggle/working',
                                  label_smoothing=label_smoothing,
                                  gen_temperature=0.8, gen_top_k=10,
                                  gen_do_sample=True)
print('SNN final train ppl:', comparison_results['snn_history']['train_ppl'][-1])
print('SNN final val ppl:', comparison_results['snn_history']['val_ppl'][-1])
print('Transformer final train ppl:', comparison_results['transformer_history']['train_ppl'][-1])
print('Transformer final val ppl:', comparison_results['transformer_history']['val_ppl'][-1])
print(json.dumps({k: v for k, v in comparison_results.items() if 'history' not in k}, indent=2))


In [ ]:
sample_text = 'ROMEO:'
sample_ids = tokenizer.encode(sample_text)[:seq_len]
sample_ids = sample_ids + [0] * (seq_len - len(sample_ids))
sample_one_hot = torch.zeros(1, seq_len, tokenizer.vocab_size)
sample_one_hot.scatter_(2, torch.tensor([sample_ids]).unsqueeze(2), 1.0)
sample_input = sample_one_hot.unsqueeze(0).repeat(time_steps, 1, 1, 1).to(device)
energy_report = quantize_energy(snn_model, sample_input, time_steps=time_steps)
print('SNN energy report:')
print(json.dumps(energy_report, indent=2))


## Generate sample text

In [ ]:
from snnai.benchmarks.generation_metrics import evaluate_generation
snn_model.eval()
prompt = 'ROMEO:'
generated = prompt
with torch.no_grad():
    for _ in range(200):
        indices = torch.tensor([tokenizer.encode(generated[-seq_len:])], dtype=torch.long).to(device)
        one_hot = torch.zeros(1, indices.size(1), tokenizer.vocab_size).to(device)
        one_hot.scatter_(2, indices.unsqueeze(2), 1.0)
        x = one_hot.unsqueeze(0).repeat(time_steps, 1, 1, 1)
        out = snn_model(x)
        idx = int(out[0, -1].argmax().item())
        generated += tokenizer.idx_to_char[idx]
print('Generated:')
print(generated)
# Sampling generation with temperature/top-k
sampled = evaluate_generation(snn_model, tokenizer, ['ROMEO:', 'JULIET:', 'The '],
                              max_chars=200, device=device,
                              temperature=0.8, top_k=10, do_sample=True)
print('Sampled generation:')
for g in sampled['generated']:
    print(g[:200])


## Optimization: quantization and pruning

In [ ]:
scales = quantize_weights(snn_model)
print('Quantized scale sample:', list(scales.items())[:2])
prune_info = prune_weights(snn_model, threshold=0.005)
print('Pruning info:', prune_info)

## Save model

In [ ]:
torch.save({
    'model_state_dict': snn_model.state_dict(),
    'vocab': vocab,
    'history': snn_history,
    'comparison': comparison_results
}, 'snnai_v6_large_lm.pt')
print('Saved snnai_v6_large_lm.pt')
